In [9]:
import numpy as np
import pandas as pd

# Task 0
Read the dataset from csv file & perform data cleaning - remove all rows, which contains `?` in some columns.
Also check for data correctness (salary & salary $K).

In [11]:
# write your code for task 0 here
df = pd.read_csv("../data/adult.csv")
df = df.replace("?", np.nan)
df.dropna(axis=0, how="any", inplace=True)

invalid_low = (df["salary K$"] > 50) & (df["salary"] == "<=50")
invalid_high = (df["salary K$"] <= 50) & (df["salary"] == ">50")

print(f"Understatement errors found: {invalid_low.sum()}")
print(f"Overstatement errors found: {invalid_high.sum()}")
df.head(n=50)

Understatement errors found: 0
Overstatement errors found: 0


,Unnamed: 0,age,workclass,education,marital-status,occupation,relationship,race,sex,hours-per-week,native-country,salary,salary K$
0,0,39,State-gov,Bachelors,Never-married,Adm-clerical,Not-in-family,White,Male,40,United-States,<=50K,39
1,1,50,Self-emp-not-inc,Bachelors,Married-civ-spouse,Exec-managerial,Husband,White,Male,13,United-States,<=50K,35
2,2,38,Private,HS-grad,Divorced,Handlers-cleaners,Not-in-family,White,Male,40,United-States,<=50K,27
3,3,53,Private,11th,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,40,United-States,<=50K,43
4,4,28,Private,Bachelors,Married-civ-spouse,Prof-specialty,Wife,Black,Female,40,Cuba,<=50K,25
5,5,37,Private,Masters,Married-civ-spouse,Exec-managerial,Wife,White,Female,40,United-States,<=50K,44
6,6,49,Private,9th,Married-spouse-absent,Other-service,Not-in-family,Black,Female,16,Jamaica,<=50K,49
7,7,52,Self-emp-not-inc,HS-grad,Married-civ-spouse,Exec-managerial,Husband,White,Male,45,United-States,>50K,307
8,8,31,Private,Masters,Never-married,Prof-specialty,Not-in-family,White,Female,50,United-States,>50K,330
9,9,42,Private,Bachelors,Married-civ-spouse,Exec-managerial,Husband,White,Male,40,United-States,>50K,303


# Task 1
Print the count of men and women in the dataset.

In [12]:
# write your code for task 1 here
print(df["sex"].value_counts())

sex
Male      20380
Female     9782
Name: count, dtype: int64


# Task 2
Find the average age of men in dataset

In [16]:
# write your code for task 2 here
male_mask = df["sex"] == "Male"
male_df = df[male_mask]
avg_male_age = male_df["age"].mean()
print(f"Average age of men is equal: {avg_male_age:.0f}")

Average age of men is equal: 39


# Task 3
Get the percentage of people from Poland (native-country)

In [19]:
# write your code for task 3 here
poland_mask = df["native-country"] == "Poland"
total_count = len(df)
poland_count = poland_mask.sum()
poland_percentage = (poland_count / total_count) * 100
print(f"Percentage of people from Poland: {poland_percentage:.2f}%")


56
Percentage of people from Poland: 0.19%


# Task 4
Get the mean and standard deviation of the age for people who earn > 50K per year. After this, get it for those who earn <= 50K.

In [20]:
# write your code for task 4 here
clean_salary = df["salary"]
result = df.groupby(clean_salary)["age"].agg(["mean", "std"])
print(result)

            mean        std
salary                     
<=50K   36.60806  13.464631
>50K    43.95911  10.269633


# Task 5
Check, if there are some people without higher education (education: Bachelors, Prof-school, Assoc-acdm, Assoc-voc, Masters, Doctorate), but with > 50K salary

In [21]:
# write your code for task 5 here
higher_ed = ['Bachelors', 'Prof-school', 'Assoc-acdm', 'Assoc-voc', 'Masters', 'Doctorate']
result = df[(~df["education"].isin(higher_ed)) & (df["salary"] == ">50K")]
print("Number of matches:", len(result))

Number of matches: 3178


# Task 6
Get the statistics of age for each type of education. Use `groupby` and `describe` for this.

In [22]:
# write your code for task 6 here
clean_education = df["education"].str.strip()
result = df.groupby(clean_education)["age"].describe()
print(result)



               count       mean        std   min   25%   50%   75%   max
education                                                               
10th           820.0  37.897561  16.225795  17.0  23.0  36.0  52.0  90.0
11th          1048.0  32.363550  15.089307  17.0  18.0  28.5  43.0  90.0
12th           377.0  32.013263  14.373710  17.0  19.0  28.0  41.0  79.0
1st-4th        151.0  44.622517  14.929051  19.0  33.0  44.0  56.0  81.0
5th-6th        288.0  41.649306  14.754622  17.0  28.0  41.0  53.0  82.0
7th-8th        557.0  47.631957  15.737479  17.0  34.0  49.0  60.0  90.0
9th            455.0  40.303297  15.335754  17.0  28.0  38.0  53.0  90.0
Assoc-acdm    1008.0  37.286706  10.509755  19.0  29.0  36.0  44.0  90.0
Assoc-voc     1307.0  38.246366  11.181253  19.0  30.0  37.0  45.0  84.0
Bachelors     5044.0  38.641554  11.577566  19.0  29.0  37.0  46.0  90.0
Doctorate      375.0  47.130667  11.471727  24.0  39.0  47.0  54.0  80.0
HS-grad       9840.0  38.640955  13.067730  17.0  2

# Task 7
Compare the married and non-married men salaries. Who earns more? (>50K or <=50K)
Married men are those, whom `marital-status` starts with "Married". Others are not.

In [26]:
# write your code for task 7 here
males = df[df["sex"] == "Male"].copy()
males["is_married"] = males["marital-status"].str.startswith("Married").replace({
    True: "Married",
    False: "Non-Married"
})
comparison = pd.crosstab(
    index=males["is_married"],
    columns=males["salary"],
    normalize="index"
) * 100
comparison.index.name = None
print(comparison.round(2))

salary       <=50K   >50K
Married      55.20  44.80
Non-Married  91.15   8.85


# Task 8
Get the max hours per week some person works. How many people works the same amount of hours per week?

In [27]:
# write your code for task 8 here
max_hours = df["hours-per-week"].max()
result = len(df[df["hours-per-week"] == max_hours])
print(f"Max hours per week: {max_hours}")
print(f"Number of people working this amount: {result}")

Max hours per week: 99
Number of people working this amount: 78


# Task 9
Analyze the correlation between data in dataset. Understand connected fields in it and print highlight thier connection.

In [31]:
# write your code for task 9 here
corr_matrix = df.corr(numeric_only=True)
print("=== Full Correlation Matrix ===")
print(corr_matrix.round(3))
print("\n")
stacked_corr = corr_matrix.unstack()
stacked_corr = stacked_corr[stacked_corr != 1.0]
strongest_correlations = stacked_corr.abs().sort_values(ascending=False)

print("The strongest connection in dataset:")
for index, value in strongest_correlations[0:6:2].items():
    col1, col2 = index
    original_corr = corr_matrix.loc[col1, col2]
    print(f"Field {col1} and {col2} have correlation: {original_corr:.3f}")

=== Full Correlation Matrix ===
                Unnamed: 0    age  hours-per-week  salary K$
Unnamed: 0           1.000 -0.001          -0.002      0.000
age                 -0.001  1.000           0.102      0.208
hours-per-week      -0.002  0.102           1.000      0.196
salary K$            0.000  0.208           0.196      1.000


The strongest connection in dataset:
Field age and salary K$ have correlation: 0.208
Field hours-per-week and salary K$ have correlation: 0.196
Field age and hours-per-week have correlation: 0.102
